# Comprehensive Causal Experiments — Detection-Expression Gap

This notebook runs the **full set of causal experiments** for the hallucination
paper:

1. **Readout bypass** (with γ sweep + precision matrix on Factual / Impossible / Hallucination)
2. **Boundary steering** (with α sweep, both directions)
3. **Manifold repair** (with λ sweep)

Plus the supporting analyses that strengthen the causal claims:

- Probe AUROC, layer-wise probe accuracy, cross-domain probe transfer
- Output visibility of the boundary direction under $W_U$ at multiple SVD cutoffs
- SVA / RSF mechanism test: boundary alignment with $\Pi_\text{pred}$ vs $\Pi_\text{input}$
- Fisher sensitivity and Hessian curvature along the boundary
- Random direction controls matched in norm
- Detection benchmark: probe AUROC vs max-logit confidence vs predictive entropy

**Resumability:** every step writes its outputs to disk. Re-running skips
already-computed artifacts unless `FORCE_RECOMPUTE=True`.

**Memory:** models are loaded sequentially. A `del model; gc.collect();
torch.cuda.empty_cache()` block runs between models. Qwen3-32B is configured to
use bf16 by default but a 4-bit toggle is available.


## 1. Setup & imports

In [ ]:
import os
import gc
import json
import time
import math
import pickle
import warnings
from pathlib import Path
from collections import defaultdict
from typing import Optional, Tuple, Dict, List

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.preprocessing import StandardScaler

from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")
torch.set_grad_enabled(False)  # default; we enable locally for Fisher/Hessian

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"device   : {DEVICE}")
print(f"dtype    : {DTYPE}")
if DEVICE == 'cuda':
    print(f"gpu      : {torch.cuda.get_device_name(0)}")
    print(f"vram     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## 2. Configuration

Edit `MODELS_TO_RUN` to control which models to process. Intervention layers
follow the paper (layer 16 for Llama-3.2-3B, layer 20 for Qwen-2.5-3B); for
other models we use a heuristic of `int(0.55 * n_layers)` which can be
overridden.

**Causal experiments** run on every model by default. Set
`CAUSAL_MODELS` to a subset if you want to skip larger models initially.


In [ ]:
# -----------------------------------------------------------------------
# Paths
# -----------------------------------------------------------------------
DATA_DIR          = Path(".")
RESULTS_DIR       = Path("./results_causal")
HIDDEN_CACHE_DIR  = Path("./cache_hidden_states")
PROBE_CACHE_DIR   = Path("./cache_probes")
WU_CACHE_DIR      = Path("./cache_wu_svd")

for d in [RESULTS_DIR, HIDDEN_CACHE_DIR, PROBE_CACHE_DIR, WU_CACHE_DIR]:
    d.mkdir(exist_ok=True)

FORCE_RECOMPUTE = False

# -----------------------------------------------------------------------
# Models
# -----------------------------------------------------------------------
# Intervention layer follows the paper for the two reference models; for
# others we use roughly 55% of depth (the empirical onset of fracture in
# the geometry results) — override per model if needed.
MODELS = {
    "llama3.2-3B": dict(
        hf_id="meta-llama/Llama-3.2-3B",
        intervention_layer=16,
        load_in_4bit=False,
    ),
    "llama3.2-3B-IT": dict(
        hf_id="meta-llama/Llama-3.2-3B-Instruct",
        intervention_layer=16,
        load_in_4bit=False,
    ),
    "llama3.1-8B": dict(
        hf_id="meta-llama/Llama-3.1-8B",
        intervention_layer=18,
        load_in_4bit=False,
    ),
    "mistralv1-7B": dict(
        hf_id="mistralai/Mistral-7B-v0.1",
        intervention_layer=18,
        load_in_4bit=False,
    ),
    "qwen2.5-7B": dict(
        hf_id="Qwen/Qwen2.5-7B",
        intervention_layer=18,
        load_in_4bit=False,
    ),
    "qwen2.5it-3B": dict(
        hf_id="Qwen/Qwen2.5-3B-Instruct",
        intervention_layer=20,
        load_in_4bit=False,
    ),
    "qwen3-8B": dict(
        hf_id="Qwen/Qwen3-8B",
        intervention_layer=20,
        load_in_4bit=False,
    ),
    "qwen3-32B": dict(
        hf_id="Qwen/Qwen3-32B",
        intervention_layer=36,
        load_in_4bit=False,  # set True if VRAM is tight
    ),
}

MODELS_TO_RUN  = list(MODELS.keys())
CAUSAL_MODELS  = list(MODELS.keys())  # causal interventions on all by default

# -----------------------------------------------------------------------
# Experiment parameters
# -----------------------------------------------------------------------
N_PER_SPLIT      = 500        # cap per split for tractable runtime
MAX_NEW_TOKENS   = 80
GEN_BATCH_SIZE   = 4          # generation batch size during interventions

# Probe
PROBE_C          = 1.0        # logistic regression regularisation

# Output visibility cutoffs
VIS_CUTOFFS      = [16, 64, 256, 1024]

# Causal sweeps
GAMMA_SWEEP      = [0.0, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0]
ALPHA_SWEEP      = [-2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0]
LAMBDA_SWEEP     = [0.0, 0.25, 0.5, 0.75, 1.0]
N_RANDOM_DIRS    = 5

# Fisher / Hessian
EPSILON          = 1e-3

print(f"models scheduled : {MODELS_TO_RUN}")
print(f"causal targets   : {CAUSAL_MODELS}")
print(f"n per split      : {N_PER_SPLIT}")


## 3. Data loading

- **Factual**: `factual.csv` (your manually-verified set)
- **Impossible**: `impossibleQA.csv` (false-premise / underspecified)
- **Hallucination**: difficult TruthfulQA + long-tail PopQA, loaded from HF

If you prefer to point at a local hallucination CSV, set
`HALLUCINATION_CSV` below.


In [ ]:
HALLUCINATION_CSV = None  # set to a path to override the HF mix

def _column(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"none of {candidates} found in {list(df.columns)}")

def load_split_csv(path):
    df = pd.read_csv(path)
    qcol = _column(df, ["question", "Question", "prompt", "text", "query"])
    questions = df[qcol].astype(str).str.strip().tolist()
    questions = [q for q in questions if len(q) > 0]
    return questions

def build_hallucination_split(n_target=N_PER_SPLIT, seed=SEED):
    """Mix difficult TruthfulQA + long-tail PopQA."""
    if HALLUCINATION_CSV is not None:
        return load_split_csv(HALLUCINATION_CSV)
    from datasets import load_dataset
    rng = np.random.RandomState(seed)
    half = n_target // 2

    tqa = load_dataset("truthful_qa", "generation", split="validation")
    tqa_qs = [item["question"] for item in tqa]
    rng.shuffle(tqa_qs)
    tqa_sel = tqa_qs[:half]

    try:
        popqa = load_dataset("akariasai/PopQA", split="test")
        items = list(popqa)
        # 's_pop' is subject popularity; lower = long tail
        items_sorted = sorted(items, key=lambda x: x.get("s_pop", 1e12))
        long_tail_pool = [it["question"] for it in items_sorted[:half * 4]]
        rng.shuffle(long_tail_pool)
        pop_sel = long_tail_pool[:half]
    except Exception as e:
        print(f"PopQA load failed ({e}); falling back to all-TruthfulQA")
        rng.shuffle(tqa_qs)
        pop_sel = tqa_qs[half:half + half]

    mix = tqa_sel + pop_sel
    rng.shuffle(mix)
    return mix[:n_target]

SPLITS = {}
SPLITS["factual"]       = load_split_csv(DATA_DIR / "factual.csv")[:N_PER_SPLIT]
SPLITS["impossible"]    = load_split_csv(DATA_DIR / "impossibleQA.csv")[:N_PER_SPLIT]
SPLITS["hallucination"] = build_hallucination_split()

for name, qs in SPLITS.items():
    print(f"{name:>14s}: {len(qs):4d} questions  | sample: {qs[0][:80]!r}")


## 4. Utility functions

Refusal detection, loop detection, prompt formatting, generation with hooks,
and the helper that finds a single-token uncertainty target per tokenizer.


In [ ]:
import re

REFUSAL_PATTERNS = [
    r"\bi don'?t know\b",
    r"\bi'?m (not sure|unsure|uncertain)\b",
    r"\bi am (not sure|unsure|uncertain)\b",
    r"\bcannot (be )?(answered?|determined?|known?)\b",
    r"\bcan'?t (be )?(answered?|determined?|known?)\b",
    r"\bunable to (answer|determine|know|provide)\b",
    r"\bno (clear|definitive|known|reliable) (answer|information)\b",
    r"\b(not|isn'?t) (possible|knowable|answerable)\b",
    r"\bimpossible to (answer|determine|know|say)\b",
    r"\binsufficient (information|data|context)\b",
    r"\b(without|need) (more|additional) (information|context|details)\b",
    r"\b(i'?m|i am) sorry\b",
    r"\b(unclear|ambiguous)\b",
    r"^\s*(i don'?t|unclear|unknown|uncertain|unsure)\b",
]
REFUSAL_RE = re.compile("|".join(REFUSAL_PATTERNS), re.IGNORECASE)

def is_refusal(text: str) -> bool:
    return bool(REFUSAL_RE.search(text or ""))

def detect_loop(text: str, n: int = 3, threshold: int = 3) -> bool:
    """Detect a repeated n-gram appearing >= threshold consecutive times."""
    if not text:
        return False
    toks = text.split()
    if len(toks) < n * threshold:
        return False
    for i in range(len(toks) - n * threshold + 1):
        ngram = tuple(toks[i:i + n])
        ok = True
        for k in range(1, threshold):
            if tuple(toks[i + k*n : i + (k+1)*n]) != ngram:
                ok = False
                break
        if ok:
            return True
    return False

def format_prompt(question: str, is_instruct: bool, tokenizer) -> str:
    """Use chat template for instruct models; raw prompt otherwise."""
    if is_instruct and tokenizer.chat_template is not None:
        msgs = [{"role": "user", "content": question}]
        return tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True
        )
    return f"Question: {question}\nAnswer:"

UNCERTAIN_TOKEN_CANDIDATES = [
    " unsure", " uncertain", " unknown",
    " I don't know", " idk", " maybe",
    "unsure", "I don't",
]

def find_uncertainty_token_id(tokenizer) -> Tuple[int, str]:
    """Return (token_id, surface) for a single-token uncertainty marker."""
    for cand in UNCERTAIN_TOKEN_CANDIDATES:
        ids = tokenizer.encode(cand, add_special_tokens=False)
        if len(ids) == 1:
            return ids[0], cand
    # Fall back: first token of " unsure"
    ids = tokenizer.encode(" unsure", add_special_tokens=False)
    return ids[0], tokenizer.decode(ids[:1])

def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

print("utility functions defined")


## 5. Model loader

Loads with the configured dtype; supports optional 4-bit quantization for very
large models. Returns the model, tokenizer, and a `meta` dict with the
intervention layer index and total layer count.


In [ ]:
def load_model(model_key: str):
    cfg = MODELS[model_key]
    print(f"loading {model_key} ({cfg['hf_id']})...")
    t0 = time.time()
    tokenizer = AutoTokenizer.from_pretrained(cfg["hf_id"])
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"  # left-pad for generation

    kwargs = dict(torch_dtype=DTYPE, device_map="auto",
                  attn_implementation="eager")
    if cfg.get("load_in_4bit", False):
        from transformers import BitsAndBytesConfig
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )
        kwargs["quantization_config"] = bnb
        kwargs.pop("torch_dtype", None)

    model = AutoModelForCausalLM.from_pretrained(cfg["hf_id"], **kwargs)
    model.eval()

    base = model.model if hasattr(model, "model") else model
    n_layers = len(base.layers) if hasattr(base, "layers") else \
               len(base.h) if hasattr(base, "h") else None
    meta = dict(
        intervention_layer=cfg["intervention_layer"],
        n_layers=n_layers,
        hf_id=cfg["hf_id"],
        is_instruct=("Instruct" in cfg["hf_id"]) or ("-IT" in cfg["hf_id"]),
        d_model=model.config.hidden_size,
    )
    print(f"  -> done in {time.time()-t0:.1f}s | layers={n_layers} | d={meta['d_model']}")
    return model, tokenizer, meta


## 6. Hidden state extraction

Extracts last-token hidden states at every layer for every question in every
split, using `output_hidden_states=True`. Caches per-model arrays.


In [ ]:
@torch.no_grad()
def extract_hidden_states(model, tokenizer, meta, questions: List[str],
                          batch_size: int = 8) -> np.ndarray:
    """Return array of shape (n_questions, n_layers+1, d_model)."""
    hs_all = []
    prompts = [format_prompt(q, meta["is_instruct"], tokenizer) for q in questions]
    for i in tqdm(range(0, len(prompts), batch_size), desc="extract", leave=False):
        chunk = prompts[i:i + batch_size]
        enc = tokenizer(chunk, return_tensors="pt", padding=True,
                        truncation=True, max_length=256).to(DEVICE)
        out = model(**enc, output_hidden_states=True, use_cache=False)
        # tuple of (n_layers+1,) tensors each (B, T, d)
        # last non-pad position per row:
        last_idx = enc.attention_mask.sum(dim=1) - 1
        per_layer = []
        for h in out.hidden_states:
            sel = h[torch.arange(h.size(0)), last_idx]  # (B, d)
            per_layer.append(sel.float().cpu().numpy())
        per_layer = np.stack(per_layer, axis=1)  # (B, L+1, d)
        hs_all.append(per_layer)
        del out
    return np.concatenate(hs_all, axis=0)

def extract_or_load_hidden_states(model_key, model, tokenizer, meta):
    cache_path = HIDDEN_CACHE_DIR / f"{model_key}_hidden.npz"
    if cache_path.exists() and not FORCE_RECOMPUTE:
        print(f"  loading cached hidden states: {cache_path.name}")
        data = np.load(cache_path)
        return {k: data[k] for k in data.files}
    hidden = {}
    for split_name, qs in SPLITS.items():
        print(f"  extracting hidden states for split={split_name}...")
        hidden[split_name] = extract_hidden_states(model, tokenizer, meta, qs)
    np.savez_compressed(cache_path, **hidden)
    return hidden


## 7. Probe training and cross-domain transfer

Trains a logistic regression probe at the intervention layer to classify
`factual` vs `impossible`. Then evaluates **cross-domain transfer** to
`hallucination`. Also fits per-layer probes for the layer-wise accuracy curve.


In [ ]:
def fit_probe(h_pos, h_neg, C=PROBE_C):
    """Fit logistic regression with h_pos = uncertain (label 1), h_neg = factual (0)."""
    X = np.concatenate([h_pos, h_neg], axis=0)
    y = np.concatenate([np.ones(len(h_pos)), np.zeros(len(h_neg))])
    scaler = StandardScaler().fit(X)
    Xs = scaler.transform(X)
    clf = LogisticRegression(C=C, max_iter=2000, random_state=SEED).fit(Xs, y)
    return clf, scaler

def eval_probe(clf, scaler, X, y_true=None):
    Xs = scaler.transform(X)
    prob_unc = clf.predict_proba(Xs)[:, 1]
    pred = (prob_unc >= 0.5).astype(int)
    out = dict(prob_uncertain=prob_unc, pred=pred)
    if y_true is not None:
        out["accuracy"] = float(accuracy_score(y_true, pred))
        if len(set(y_true)) > 1:
            out["auroc"] = float(roc_auc_score(y_true, prob_unc))
    return out

def train_probes(hidden, meta, model_key):
    """Train probes at all layers; cache."""
    cache_path = PROBE_CACHE_DIR / f"{model_key}_probes.pkl"
    if cache_path.exists() and not FORCE_RECOMPUTE:
        print(f"  loading cached probes: {cache_path.name}")
        return pickle.loads(cache_path.read_bytes())

    n_layers = hidden["factual"].shape[1]
    results = {"per_layer": {}}
    rng = np.random.RandomState(SEED)
    n_min = min(len(hidden["factual"]), len(hidden["impossible"]))
    idx = rng.permutation(n_min)
    train_idx, test_idx = idx[: int(0.8 * n_min)], idx[int(0.8 * n_min):]

    for layer in range(n_layers):
        Hf = hidden["factual"][:n_min, layer]
        Hi = hidden["impossible"][:n_min, layer]
        Hh = hidden["hallucination"][:, layer]

        clf, scaler = fit_probe(Hi[train_idx], Hf[train_idx])

        # in-domain test
        X_test = np.concatenate([Hi[test_idx], Hf[test_idx]])
        y_test = np.concatenate([np.ones(len(test_idx)), np.zeros(len(test_idx))])
        in_eval = eval_probe(clf, scaler, X_test, y_test)

        # cross-domain: hallucination → uncertain side
        cross_eval = eval_probe(clf, scaler, Hh)

        results["per_layer"][layer] = dict(
            in_domain_acc=in_eval.get("accuracy"),
            in_domain_auroc=in_eval.get("auroc"),
            hall_uncertain_rate=float(cross_eval["pred"].mean()),
            hall_mean_prob=float(cross_eval["prob_uncertain"].mean()),
        )

    # Save the intervention-layer probe in full for downstream causal use
    L = meta["intervention_layer"]
    n_min = min(len(hidden["factual"]), len(hidden["impossible"]))
    Hf = hidden["factual"][:n_min, L]
    Hi = hidden["impossible"][:n_min, L]
    clf, scaler = fit_probe(Hi, Hf)
    results["intervention_probe"] = dict(clf=clf, scaler=scaler, layer=L)

    cache_path.write_bytes(pickle.dumps(results))
    return results


## 8. Boundary direction, output visibility, and SVA mechanism test

Three things in one cell:

1. **Boundary direction** $b_l = (\mu^U - \mu^F) / \|\mu^U - \mu^F\|$ at every layer.
2. **Output visibility** $\text{vis}_m(b_l)$ compared to $\text{vis}_m$ of
   factual-prediction directions and random directions, at multiple cutoffs.
3. **SVA mechanism test**: the boundary's RSF in the **prediction manifold**
   $\Pi_\text{pred}$ (row space of $W_U$) versus the **input subspace**
   $\Pi_\text{input}$ (principal subspace of intervention-layer activations).
   The hypothesis predicts the boundary lies preferentially in $\Pi_\text{input}$.

We use `torch.pca_lowrank` to avoid the LAPACK issues we hit before.


In [ ]:
def compute_boundary(hidden, layer):
    mu_u = hidden["impossible"][:, layer].mean(axis=0)
    mu_f = hidden["factual"][:, layer].mean(axis=0)
    diff = mu_u - mu_f
    norm = np.linalg.norm(diff)
    return diff / (norm + 1e-12), float(norm)

@torch.no_grad()
def get_wu(model):
    """Return W_U as (V, d) float32 cpu tensor."""
    lm_head = model.get_output_embeddings()
    W = lm_head.weight.detach().float().cpu()  # (V, d)
    return W

@torch.no_grad()
def wu_top_singular_directions(W_U, k=1024):
    """Top-k right singular vectors of W_U via low-rank SVD."""
    # torch.svd_lowrank operates on (m, n) returning U,S,V where A ≈ U diag(S) V^T
    # We want the top right singular vectors (columns of V).
    A = W_U  # (V, d)
    q = min(k + 16, A.shape[1])
    U, S, V = torch.svd_lowrank(A, q=q, niter=4)
    # take top-k
    V_top = V[:, :k].contiguous()  # (d, k)
    S_top = S[:k]
    return V_top.numpy(), S_top.numpy()

def visibility(direction, V_top, m):
    """vis_m(u) = ||P_{V_m} u|| / ||u||  where V_m = span of first m cols."""
    Vm = V_top[:, :m]                       # (d, m)
    u  = direction / (np.linalg.norm(direction) + 1e-12)
    proj = Vm @ (Vm.T @ u)
    return float(np.linalg.norm(proj))

def rsf(direction, basis):
    """Relative subspace fraction: ||P_basis u||^2 / ||u||^2 for orthonormal basis."""
    u = direction / (np.linalg.norm(direction) + 1e-12)
    proj = basis @ (basis.T @ u)
    return float(np.linalg.norm(proj) ** 2)

def boundary_visibility_and_rsf(hidden, meta, W_U, V_top, model_key):
    out = {}
    L = meta["intervention_layer"]

    # --- 1. Boundary at every layer + at intervention layer ---
    n_layers = hidden["factual"].shape[1]
    boundary_per_layer, norms_per_layer = [], []
    for layer in range(n_layers):
        b, n = compute_boundary(hidden, layer)
        boundary_per_layer.append(b)
        norms_per_layer.append(n)
    out["boundary_norm_per_layer"] = norms_per_layer
    b_L = boundary_per_layer[L]

    # --- 2. Factual prediction directions (top-k unembedding rows of likely tokens) ---
    # Use mean factual hidden state -> top-5 predicted tokens -> their W_U rows.
    mu_f = hidden["factual"][:, L].mean(axis=0)
    logits_proxy = W_U.numpy() @ mu_f
    top_tokens = np.argsort(-logits_proxy)[:5]
    factual_dirs = W_U.numpy()[top_tokens]
    factual_dirs /= (np.linalg.norm(factual_dirs, axis=1, keepdims=True) + 1e-12)

    # --- 3. Random directions matched in dimension ---
    rng = np.random.RandomState(SEED)
    rand_dirs = rng.randn(N_RANDOM_DIRS, b_L.shape[0]).astype(np.float32)
    rand_dirs /= np.linalg.norm(rand_dirs, axis=1, keepdims=True)

    # --- 4. Visibility at multiple cutoffs ---
    vis_table = []
    for m in VIS_CUTOFFS:
        vis_table.append(dict(
            cutoff=m,
            boundary=visibility(b_L, V_top, m),
            factual_pred_mean=float(np.mean([visibility(d, V_top, m) for d in factual_dirs])),
            random_mean=float(np.mean([visibility(d, V_top, m) for d in rand_dirs])),
            random_std=float(np.std([visibility(d, V_top, m) for d in rand_dirs])),
        ))
    out["visibility_table"] = vis_table

    # --- 5. SVA / RSF mechanism test ---
    # Pi_pred = orthonormal basis of top-r right singular vectors of W_U.
    # Pi_input = top-r PCA components of (centered) factual+impossible hidden states at L.
    r = 256
    Pi_pred = V_top[:, :r]                              # (d, r) orthonormal
    H = np.concatenate([hidden["factual"][:, L],
                        hidden["impossible"][:, L]], axis=0)
    H = H - H.mean(axis=0, keepdims=True)
    Ht = torch.from_numpy(H).float()
    Ur, Sr, Vr = torch.svd_lowrank(Ht, q=r + 16, niter=4)
    Pi_input = Vr[:, :r].numpy()                        # (d, r) orthonormal

    out["rsf_boundary_in_pi_pred"]  = rsf(b_L, Pi_pred)
    out["rsf_boundary_in_pi_input"] = rsf(b_L, Pi_input)
    out["rsf_ratio_input_over_pred"] = (
        out["rsf_boundary_in_pi_input"] /
        max(out["rsf_boundary_in_pi_pred"], 1e-12)
    )

    # Also for factual prediction directions and random (for context)
    out["rsf_factual_pred_in_pi_pred"]  = float(np.mean([rsf(d, Pi_pred) for d in factual_dirs]))
    out["rsf_factual_pred_in_pi_input"] = float(np.mean([rsf(d, Pi_input) for d in factual_dirs]))
    out["rsf_random_in_pi_pred"]  = float(np.mean([rsf(d, Pi_pred) for d in rand_dirs]))
    out["rsf_random_in_pi_input"] = float(np.mean([rsf(d, Pi_input) for d in rand_dirs]))

    out["boundary_intervention_layer"] = b_L
    return out


## 9. Fisher sensitivity and Hessian curvature along the boundary

Local probes of how much the output distribution changes when you move along
$b_l$ at the intervention layer. We use the symmetric-KL approximation for
Fisher and a 3-point stencil for Hessian curvature.


In [ ]:
@torch.no_grad()
def fisher_hessian_along(model, tokenizer, meta, questions, boundary, epsilon=EPSILON):
    L = meta["intervention_layer"]
    base = model.model if hasattr(model, "model") else model
    layer_mod = base.layers[L] if hasattr(base, "layers") else base.h[L]

    b_t = torch.from_numpy(boundary).to(DEVICE).to(DTYPE)

    fishers, hessians = [], []
    inject = {"mode": None}

    def hook(module, inp, out):
        h = out[0] if isinstance(out, tuple) else out
        if inject["mode"] is not None:
            # h: (B, T, d) — apply at last token only
            sign = inject["mode"]
            h = h.clone()
            h[:, -1, :] = h[:, -1, :] + sign * epsilon * b_t
            if isinstance(out, tuple):
                return (h,) + out[1:]
            return h
        return out

    handle = layer_mod.register_forward_hook(hook)
    try:
        for q in tqdm(questions[:50], desc="fisher", leave=False):
            prompt = format_prompt(q, meta["is_instruct"], tokenizer)
            enc = tokenizer(prompt, return_tensors="pt").to(DEVICE)

            inject["mode"] = None
            out0 = model(**enc, use_cache=False).logits[0, -1].float()
            p0 = F.softmax(out0, dim=-1)
            logp0 = F.log_softmax(out0, dim=-1)

            inject["mode"] = +1
            outp = model(**enc, use_cache=False).logits[0, -1].float()
            pp = F.softmax(outp, dim=-1); logpp = F.log_softmax(outp, dim=-1)

            inject["mode"] = -1
            outm = model(**enc, use_cache=False).logits[0, -1].float()
            pm = F.softmax(outm, dim=-1); logpm = F.log_softmax(outm, dim=-1)

            sym_kl = 0.5 * ((p0 * (logp0 - logpp)).sum() +
                            (pp * (logpp - logp0)).sum())
            fisher = sym_kl.item() / (epsilon ** 2)

            # Hessian of -log p(argmax_token) along b
            tok = out0.argmax().item()
            L0 = -logp0[tok].item()
            Lp = -logpp[tok].item()
            Lm = -logpm[tok].item()
            hess = (Lp - 2 * L0 + Lm) / (epsilon ** 2)

            fishers.append(fisher)
            hessians.append(hess)
    finally:
        handle.remove()
        inject["mode"] = None

    return dict(
        fisher_mean=float(np.mean(fishers)),
        fisher_std =float(np.std(fishers)),
        hessian_mean=float(np.mean(hessians)),
        hessian_std =float(np.std(hessians)),
    )


## 10. Causal interventions

Three interventions share a generation harness that supports a per-layer
hidden-state hook plus an optional logit modifier. All three run on **all
three splits** (Factual / Impossible / Hallucination) so we get the precision
matrix the paper needs.

- **Readout bypass**: γ × probe_p(uncertain) added to the uncertainty token's
  logit at every generation step.
- **Boundary steering**: $h_L \leftarrow h_L + \alpha v_\text{steer}$ where
  $v_\text{steer} = \mu^U - \mu^F$.
- **Manifold repair**: project hidden state onto factual PCA subspace at
  intervention layer.


In [ ]:
class InterventionHarness:
    """Registers a hidden-state hook at intervention layer; supports several modes."""
    def __init__(self, model, meta):
        self.model = model
        self.meta = meta
        self.L = meta["intervention_layer"]
        base = model.model if hasattr(model, "model") else model
        self.layer_mod = base.layers[self.L] if hasattr(base, "layers") else base.h[self.L]
        self.config = {"mode": None}
        self._handle = None

    def _hook(self, module, inp, out):
        h = out[0] if isinstance(out, tuple) else out
        if self.config["mode"] is None:
            return out
        # apply only on the last (newest) token position so that during
        # autoregressive generation we don't keep re-perturbing past tokens.
        # we operate on a clone to be safe.
        h_new = h.clone()
        T = h_new.shape[1]
        if self.config["mode"] == "steer":
            v = self.config["v"]
            h_new[:, T-1, :] = h_new[:, T-1, :] + self.config["alpha"] * v
        elif self.config["mode"] == "repair":
            Pf = self.config["Pf"]      # (d, r) projection basis (orthonormal)
            mu = self.config["mu_f"]    # (d,)
            lam = self.config["lambda"]
            h_last = h_new[:, T-1, :].float()
            centered = h_last - mu
            proj = centered @ Pf @ Pf.T
            repaired = proj + mu
            h_last_new = (1 - lam) * h_last + lam * repaired
            h_new[:, T-1, :] = h_last_new.to(h_new.dtype)
        if isinstance(out, tuple):
            return (h_new,) + out[1:]
        return h_new

    def activate(self, **config):
        self.config = config
        if self._handle is None:
            self._handle = self.layer_mod.register_forward_hook(self._hook)

    def deactivate(self):
        if self._handle is not None:
            self._handle.remove()
            self._handle = None
        self.config = {"mode": None}


@torch.no_grad()
def generate_with_intervention(model, tokenizer, meta, prompts,
                               harness: Optional[InterventionHarness] = None,
                               logit_modifier=None,
                               max_new_tokens=MAX_NEW_TOKENS):
    """Generate text with optional layer hook (set on harness) and logit hook.

    `logit_modifier` is a callable(logits, step_idx, batch_idx) -> logits.
    Returns list of generated strings (excluding prompt).
    """
    enc = tokenizer(prompts, return_tensors="pt", padding=True,
                    truncation=True, max_length=256).to(DEVICE)
    input_ids = enc.input_ids
    attn = enc.attention_mask
    B = input_ids.size(0)

    # Track which sequences have finished
    eos = tokenizer.eos_token_id
    finished = torch.zeros(B, dtype=torch.bool, device=DEVICE)
    generated_ids = [[] for _ in range(B)]

    past_key_values = None
    cur_input = input_ids
    cur_attn = attn
    for step in range(max_new_tokens):
        out = model(input_ids=cur_input, attention_mask=cur_attn,
                    past_key_values=past_key_values, use_cache=True)
        past_key_values = out.past_key_values
        logits = out.logits[:, -1, :]  # (B, V)

        if logit_modifier is not None:
            logits = logit_modifier(logits, step)

        next_tok = logits.argmax(dim=-1)               # greedy
        next_tok = torch.where(finished, torch.tensor(eos, device=DEVICE), next_tok)
        for b in range(B):
            if not finished[b].item():
                generated_ids[b].append(next_tok[b].item())
        finished = finished | (next_tok == eos)

        cur_input = next_tok.unsqueeze(-1)
        cur_attn  = torch.cat([cur_attn, torch.ones_like(cur_input)], dim=1)
        if finished.all():
            break

    return [tokenizer.decode(ids, skip_special_tokens=True) for ids in generated_ids]


def behavior_metrics(texts):
    """Aggregate behavioral metrics over a list of generations."""
    return dict(
        refusal_rate=float(np.mean([is_refusal(t) for t in texts])),
        loop_rate=float(np.mean([detect_loop(t) for t in texts])),
        mean_length=float(np.mean([len(t.split()) for t in texts])),
    )


## 11. Readout bypass — γ sweep + precision matrix

For each split (Factual / Impossible / Hallucination), runs the bypass at
every value of γ. This produces the **precision matrix**: bypass should refuse
near-100% on Impossible and Hallucination but stay low on Factual.

We compute a prefill pass per prompt to get the intervention-layer hidden
state, run the probe, then generate with a logit modifier that adds
γ · p(uncertain) to the uncertainty token's logit at every generation step.


In [ ]:
@torch.no_grad()
def get_intervention_hidden_states_batch(model, meta, tokenizer, prompts):
    L = meta["intervention_layer"]
    enc = tokenizer(prompts, return_tensors="pt", padding=True,
                    truncation=True, max_length=256).to(DEVICE)
    out = model(**enc, output_hidden_states=True, use_cache=False)
    last_idx = enc.attention_mask.sum(dim=1) - 1
    H = out.hidden_states[L][torch.arange(enc.input_ids.size(0)), last_idx]
    return H.float().cpu().numpy()

def run_readout_bypass(model, tokenizer, meta, probe_results, model_key,
                       gammas=GAMMA_SWEEP, n_per_split=120):
    """Run bypass on all 3 splits at all gamma values."""
    clf      = probe_results["intervention_probe"]["clf"]
    scaler   = probe_results["intervention_probe"]["scaler"]
    unc_id, unc_str = find_uncertainty_token_id(tokenizer)
    print(f"  uncertainty token: {unc_str!r} (id={unc_id})")

    results = []
    for split_name in ["factual", "impossible", "hallucination"]:
        questions = SPLITS[split_name][:n_per_split]
        prompts = [format_prompt(q, meta["is_instruct"], tokenizer) for q in questions]

        # Prefill: probe scores for each prompt
        probe_probs = []
        for i in range(0, len(prompts), GEN_BATCH_SIZE):
            batch = prompts[i:i + GEN_BATCH_SIZE]
            H = get_intervention_hidden_states_batch(model, meta, tokenizer, batch)
            p = clf.predict_proba(scaler.transform(H))[:, 1]
            probe_probs.extend(p.tolist())
        probe_probs = np.array(probe_probs)

        for gamma in gammas:
            texts = []
            for i in tqdm(range(0, len(prompts), GEN_BATCH_SIZE),
                          desc=f"bypass {split_name} γ={gamma}", leave=False):
                batch_prompts = prompts[i:i + GEN_BATCH_SIZE]
                batch_p = probe_probs[i:i + GEN_BATCH_SIZE]
                bias = torch.tensor(gamma * batch_p, device=DEVICE, dtype=torch.float32)

                def modifier(logits, step, _bias=bias, _id=unc_id):
                    logits[:, _id] = logits[:, _id] + _bias.to(logits.dtype)
                    return logits

                gen = generate_with_intervention(
                    model, tokenizer, meta, batch_prompts,
                    harness=None, logit_modifier=modifier,
                )
                texts.extend(gen)

            metrics = behavior_metrics(texts)
            results.append(dict(
                model=model_key, split=split_name, gamma=float(gamma),
                mean_probe_prob=float(probe_probs.mean()),
                n=len(texts), **metrics,
                samples=texts[:3],
            ))
            print(f"    {split_name} γ={gamma:>5.1f}  refusal={metrics['refusal_rate']:.3f}  "
                  f"loop={metrics['loop_rate']:.3f}")

    return results


## 12. Boundary steering — α sweep

Adds $\alpha v_\text{steer}$ to the residual stream at the intervention layer
at every generated token. We sweep α positive (push factual states toward
uncertain) and negative (push uncertain states toward factual), and run on all
three splits.


In [ ]:
def run_boundary_steering(model, tokenizer, meta, hidden, model_key,
                          alphas=ALPHA_SWEEP, n_per_split=120):
    L = meta["intervention_layer"]
    mu_u = hidden["impossible"][:, L].mean(axis=0)
    mu_f = hidden["factual"][:, L].mean(axis=0)
    v_steer = torch.from_numpy((mu_u - mu_f).astype(np.float32)).to(DEVICE).to(DTYPE)
    print(f"  ||v_steer|| = {float(torch.norm(v_steer)):.3f}")

    harness = InterventionHarness(model, meta)
    results = []

    # Baseline (alpha=0) once per split for argmax comparison
    baseline_outputs = {}
    for split_name in ["factual", "impossible", "hallucination"]:
        prompts = [format_prompt(q, meta["is_instruct"], tokenizer)
                   for q in SPLITS[split_name][:n_per_split]]
        harness.deactivate()
        texts = []
        for i in range(0, len(prompts), GEN_BATCH_SIZE):
            texts.extend(generate_with_intervention(
                model, tokenizer, meta, prompts[i:i + GEN_BATCH_SIZE],
            ))
        baseline_outputs[split_name] = texts

    for split_name in ["factual", "impossible", "hallucination"]:
        prompts = [format_prompt(q, meta["is_instruct"], tokenizer)
                   for q in SPLITS[split_name][:n_per_split]]
        base_texts = baseline_outputs[split_name]
        for alpha in alphas:
            harness.activate(mode="steer", alpha=float(alpha), v=v_steer)
            texts = []
            for i in tqdm(range(0, len(prompts), GEN_BATCH_SIZE),
                          desc=f"steer {split_name} α={alpha}", leave=False):
                texts.extend(generate_with_intervention(
                    model, tokenizer, meta, prompts[i:i + GEN_BATCH_SIZE],
                    harness=harness,
                ))
            harness.deactivate()
            metrics = behavior_metrics(texts)
            change = float(np.mean([
                (b.strip()[:60] != t.strip()[:60]) for b, t in zip(base_texts, texts)
            ]))
            results.append(dict(
                model=model_key, split=split_name, alpha=float(alpha),
                n=len(texts), output_change=change, **metrics,
                samples=texts[:3],
            ))
            print(f"    {split_name} α={alpha:+5.2f}  change={change:.3f}  "
                  f"refusal={metrics['refusal_rate']:.3f}  loop={metrics['loop_rate']:.3f}")
    return results


## 13. Manifold repair — λ sweep

Projects uncertain hidden states at the intervention layer onto the factual
PCA subspace (95% variance), then linearly interpolates back. λ=0 leaves the
state alone, λ=1 fully projects. We run this on Impossible and Hallucination
inputs only (projecting factual onto factual is the null condition).


In [ ]:
def fit_factual_pca(hidden, layer, var_threshold=0.95):
    H = hidden["factual"][:, layer]
    H = H - H.mean(axis=0, keepdims=True)
    Ht = torch.from_numpy(H).float()
    q = min(H.shape[1], 1024)
    U, S, V = torch.svd_lowrank(Ht, q=q, niter=4)
    var = (S ** 2).numpy()
    var_ratio = var / var.sum()
    cumvar = np.cumsum(var_ratio)
    r = int(np.searchsorted(cumvar, var_threshold) + 1)
    return V[:, :r].numpy(), r

def run_manifold_repair(model, tokenizer, meta, hidden, model_key,
                        lambdas=LAMBDA_SWEEP, n_per_split=120):
    L = meta["intervention_layer"]
    Pf, r = fit_factual_pca(hidden, L, var_threshold=0.95)
    print(f"  factual PCA: {r} dims for 95% variance")
    mu_f = hidden["factual"][:, L].mean(axis=0)
    Pf_t = torch.from_numpy(Pf).to(DEVICE).to(DTYPE)
    mu_f_t = torch.from_numpy(mu_f).to(DEVICE).to(DTYPE)

    harness = InterventionHarness(model, meta)
    results = []

    for split_name in ["impossible", "hallucination"]:
        prompts = [format_prompt(q, meta["is_instruct"], tokenizer)
                   for q in SPLITS[split_name][:n_per_split]]
        for lam in lambdas:
            if lam == 0.0:
                harness.deactivate()
            else:
                harness.activate(mode="repair", Pf=Pf_t,
                                 mu_f=mu_f_t, **{"lambda": float(lam)})
            texts = []
            for i in tqdm(range(0, len(prompts), GEN_BATCH_SIZE),
                          desc=f"repair {split_name} λ={lam}", leave=False):
                texts.extend(generate_with_intervention(
                    model, tokenizer, meta, prompts[i:i + GEN_BATCH_SIZE],
                    harness=harness if lam > 0 else None,
                ))
            harness.deactivate()
            metrics = behavior_metrics(texts)
            results.append(dict(
                model=model_key, split=split_name, **{"lambda": float(lam)},
                pca_dim=r, n=len(texts), **metrics,
                samples=texts[:3],
            ))
            print(f"    {split_name} λ={lam:.2f}  refusal={metrics['refusal_rate']:.3f}  "
                  f"loop={metrics['loop_rate']:.3f}")
    return results


## 14. Random direction control

Same protocol as boundary steering, but the steering vector is a random unit
vector scaled to match $\|v_\text{steer}\|$. Run on the hallucination split
only (matching the paper's main controls table). Multiple random seeds.


In [ ]:
def run_random_controls(model, tokenizer, meta, hidden, model_key,
                        alphas=[1.0], n_dirs=N_RANDOM_DIRS, n_per_split=80):
    L = meta["intervention_layer"]
    mu_u = hidden["impossible"][:, L].mean(axis=0)
    mu_f = hidden["factual"][:, L].mean(axis=0)
    target_norm = np.linalg.norm(mu_u - mu_f)
    rng = np.random.RandomState(SEED)
    harness = InterventionHarness(model, meta)
    results = []

    prompts = [format_prompt(q, meta["is_instruct"], tokenizer)
               for q in SPLITS["hallucination"][:n_per_split]]

    # Baseline
    harness.deactivate()
    base_texts = []
    for i in range(0, len(prompts), GEN_BATCH_SIZE):
        base_texts.extend(generate_with_intervention(
            model, tokenizer, meta, prompts[i:i + GEN_BATCH_SIZE],
        ))

    for k in range(n_dirs):
        v = rng.randn(mu_u.shape[0]).astype(np.float32)
        v = v / np.linalg.norm(v) * target_norm
        v_t = torch.from_numpy(v).to(DEVICE).to(DTYPE)
        for alpha in alphas:
            harness.activate(mode="steer", alpha=float(alpha), v=v_t)
            texts = []
            for i in tqdm(range(0, len(prompts), GEN_BATCH_SIZE),
                          desc=f"rand-ctrl dir{k} α={alpha}", leave=False):
                texts.extend(generate_with_intervention(
                    model, tokenizer, meta, prompts[i:i + GEN_BATCH_SIZE],
                    harness=harness,
                ))
            harness.deactivate()
            metrics = behavior_metrics(texts)
            change = float(np.mean([
                (b.strip()[:60] != t.strip()[:60]) for b, t in zip(base_texts, texts)
            ]))
            results.append(dict(
                model=model_key, dir_idx=k, alpha=float(alpha),
                n=len(texts), output_change=change, **metrics,
            ))
    return results


## 15. Detection benchmark — probe vs max-logit vs entropy

Compares three hallucination-detection signals on the same set:

- **Probe**: $p(\text{uncertain} \mid h_L)$ from the linear probe.
- **Max-logit confidence**: $\max_v p(v)$ at the first generated step.
- **Predictive entropy**: $-\sum_v p(v) \log p(v)$ at the first generated step.

Label = 1 for `hallucination`+`impossible`, 0 for `factual`. Report AUROC.


In [ ]:
@torch.no_grad()
def compute_output_signals(model, tokenizer, meta, questions, n=200):
    """Compute max-prob and entropy at first generation step."""
    prompts = [format_prompt(q, meta["is_instruct"], tokenizer) for q in questions[:n]]
    max_probs, entropies = [], []
    for i in tqdm(range(0, len(prompts), GEN_BATCH_SIZE), desc="signals", leave=False):
        batch = prompts[i:i + GEN_BATCH_SIZE]
        enc = tokenizer(batch, return_tensors="pt", padding=True,
                        truncation=True, max_length=256).to(DEVICE)
        out = model(**enc, use_cache=False)
        last_idx = enc.attention_mask.sum(dim=1) - 1
        logits = out.logits[torch.arange(out.logits.size(0)), last_idx].float()
        probs = F.softmax(logits, dim=-1)
        max_probs.extend(probs.max(dim=-1).values.cpu().numpy().tolist())
        ent = -(probs * torch.log(probs + 1e-12)).sum(dim=-1)
        entropies.extend(ent.cpu().numpy().tolist())
    return np.array(max_probs), np.array(entropies)

def detection_benchmark(model, tokenizer, meta, hidden, probe_results, model_key, n=200):
    L = meta["intervention_layer"]
    clf = probe_results["intervention_probe"]["clf"]
    scaler = probe_results["intervention_probe"]["scaler"]

    out_signals = {}
    for split_name, qs in SPLITS.items():
        max_p, ent = compute_output_signals(model, tokenizer, meta, qs, n=n)
        H = hidden[split_name][:n, L]
        probe_p = clf.predict_proba(scaler.transform(H))[:, 1]
        out_signals[split_name] = dict(
            max_prob=max_p, entropy=ent, probe_p=probe_p[:len(max_p)],
        )

    rows = []
    # Setup A: hallucination + impossible vs factual
    for setup, pos in [("hall_vs_fact", ["hallucination"]),
                       ("imp_vs_fact",  ["impossible"]),
                       ("hall+imp_vs_fact", ["hallucination", "impossible"])]:
        y, scores_probe, scores_low_conf, scores_high_ent = [], [], [], []
        for split in pos:
            d = out_signals[split]
            n_s = len(d["probe_p"])
            y.extend([1]*n_s)
            scores_probe.extend(d["probe_p"].tolist())
            scores_low_conf.extend((-d["max_prob"]).tolist())  # higher = more uncertain
            scores_high_ent.extend(d["entropy"].tolist())
        d = out_signals["factual"]
        n_s = len(d["probe_p"])
        y.extend([0]*n_s)
        scores_probe.extend(d["probe_p"].tolist())
        scores_low_conf.extend((-d["max_prob"]).tolist())
        scores_high_ent.extend(d["entropy"].tolist())

        rows.append(dict(
            model=model_key, setup=setup,
            auroc_probe=float(roc_auc_score(y, scores_probe)),
            auroc_inv_maxprob=float(roc_auc_score(y, scores_low_conf)),
            auroc_entropy=float(roc_auc_score(y, scores_high_ent)),
        ))
    return rows


## 16. Per-model driver

Runs the full pipeline for a single model: load → extract → probe → boundary &
visibility & RSF → Fisher/Hessian → all 3 causal interventions → random
control → detection benchmark → save. Then unloads.


In [ ]:
def process_model(model_key, run_causal=True):
    out_path = RESULTS_DIR / f"{model_key}_results.pkl"
    if out_path.exists() and not FORCE_RECOMPUTE:
        print(f"=== {model_key}: already done, loading ===")
        return pickle.loads(out_path.read_bytes())

    print(f"\n{'='*70}\n=== {model_key} ===\n{'='*70}")
    model, tokenizer, meta = load_model(model_key)
    results = dict(model=model_key, meta=meta)

    try:
        # ----- Hidden states -----
        print("[step 1/8] hidden state extraction")
        hidden = extract_or_load_hidden_states(model_key, model, tokenizer, meta)

        # ----- Probes -----
        print("[step 2/8] probe training")
        probes = train_probes(hidden, meta, model_key)
        results["probes"] = {
            "per_layer": probes["per_layer"],
            "intervention_layer": meta["intervention_layer"],
        }

        # ----- Boundary + visibility + RSF -----
        print("[step 3/8] boundary, visibility, RSF")
        W_U = get_wu(model)
        V_top, S_top = wu_top_singular_directions(W_U, k=max(VIS_CUTOFFS))
        vis_rsf = boundary_visibility_and_rsf(hidden, meta, W_U, V_top, model_key)
        results["visibility_rsf"] = {k: v for k, v in vis_rsf.items()
                                     if k != "boundary_intervention_layer"}
        np.save(WU_CACHE_DIR / f"{model_key}_singvals.npy", S_top)

        # ----- Fisher / Hessian -----
        print("[step 4/8] Fisher & Hessian along boundary")
        boundary_L = vis_rsf["boundary_intervention_layer"]
        results["fisher_hessian"] = {}
        for split_name in ["factual", "impossible", "hallucination"]:
            fh = fisher_hessian_along(model, tokenizer, meta,
                                      SPLITS[split_name][:50], boundary_L)
            results["fisher_hessian"][split_name] = fh
            print(f"    {split_name}: Fisher={fh['fisher_mean']:.4f}  "
                  f"Hess={fh['hessian_mean']:.4f}")

        if run_causal and model_key in CAUSAL_MODELS:
            # ----- Readout bypass -----
            print("[step 5/8] readout bypass (γ sweep + precision matrix)")
            results["bypass"] = run_readout_bypass(
                model, tokenizer, meta, probes, model_key
            )

            # ----- Boundary steering -----
            print("[step 6/8] boundary steering (α sweep)")
            results["steering"] = run_boundary_steering(
                model, tokenizer, meta, hidden, model_key
            )

            # ----- Manifold repair -----
            print("[step 7/8] manifold repair (λ sweep)")
            results["repair"] = run_manifold_repair(
                model, tokenizer, meta, hidden, model_key
            )

            # ----- Random controls -----
            print("[step 7b/8] random direction controls")
            results["random_control"] = run_random_controls(
                model, tokenizer, meta, hidden, model_key
            )

        # ----- Detection benchmark -----
        print("[step 8/8] detection AUROC benchmark")
        results["detection"] = detection_benchmark(
            model, tokenizer, meta, hidden, probes, model_key
        )

        out_path.write_bytes(pickle.dumps(results))
        print(f"saved -> {out_path}")
    finally:
        del model, tokenizer
        free_memory()

    return results


## 17. Run everything

Sequentially processes every model in `MODELS_TO_RUN`. If you only want to
re-run causal experiments (and keep cached hidden states / probes), set
`FORCE_RECOMPUTE = False` (default) — the heavy artifacts are reused.


In [ ]:
all_results = {}
for model_key in MODELS_TO_RUN:
    try:
        all_results[model_key] = process_model(model_key, run_causal=True)
    except Exception as e:
        print(f"!! {model_key} failed: {e}")
        import traceback; traceback.print_exc()
        free_memory()


## 18. Aggregation — paper-ready tables

Builds the tables you'll paste into the paper:

- `tab_bypass_precision`: bypass refusal at the chosen γ across F / I / H.
- `tab_steering`: output change & refusal at α=1 across splits.
- `tab_repair`: loop rate before/after repair across splits.
- `tab_visibility`: boundary visibility vs factual-prediction vs random.
- `tab_rsf`: boundary RSF in $\Pi_\text{pred}$ vs $\Pi_\text{input}$.
- `tab_fisher_hessian`: Fisher / Hessian per split.
- `tab_detection`: AUROC of probe vs max-prob vs entropy.
- `tab_random_control`: random-direction control change rates.


In [ ]:
def df_or_empty(rows): return pd.DataFrame(rows) if rows else pd.DataFrame()

# Reload from disk in case the notebook was restarted
all_results = {}
for p in sorted(RESULTS_DIR.glob("*_results.pkl")):
    all_results[p.stem.replace("_results", "")] = pickle.loads(p.read_bytes())

# --- bypass precision matrix ---
bypass_rows = []
for mk, res in all_results.items():
    for r in res.get("bypass", []):
        row = {k: r[k] for k in ["model","split","gamma","refusal_rate","loop_rate","mean_length","mean_probe_prob","n"]}
        bypass_rows.append(row)
tab_bypass = df_or_empty(bypass_rows)
if not tab_bypass.empty:
    print("=== BYPASS — refusal rate by (model, split, γ) ===")
    pivot = tab_bypass.pivot_table(index=["model","gamma"], columns="split",
                                   values="refusal_rate")
    print(pivot.round(3).to_string())
tab_bypass.to_csv(RESULTS_DIR / "table_bypass.csv", index=False)

# --- steering ---
steer_rows = []
for mk, res in all_results.items():
    for r in res.get("steering", []):
        steer_rows.append({k: r[k] for k in
            ["model","split","alpha","output_change","refusal_rate","loop_rate","mean_length"]})
tab_steer = df_or_empty(steer_rows)
if not tab_steer.empty:
    print("\n=== STEERING — output change by (model, α) on hallucination ===")
    sub = tab_steer[tab_steer["split"]=="hallucination"]
    print(sub.pivot_table(index="model", columns="alpha",
                          values="output_change").round(3).to_string())
tab_steer.to_csv(RESULTS_DIR / "table_steering.csv", index=False)

# --- repair ---
repair_rows = []
for mk, res in all_results.items():
    for r in res.get("repair", []):
        repair_rows.append({k: r[k] for k in
            ["model","split","lambda","refusal_rate","loop_rate","mean_length","pca_dim"]})
tab_repair = df_or_empty(repair_rows)
if not tab_repair.empty:
    print("\n=== REPAIR — loop rate by (model, λ) ===")
    print(tab_repair.pivot_table(index=["model","split"], columns="lambda",
                                 values="loop_rate").round(3).to_string())
tab_repair.to_csv(RESULTS_DIR / "table_repair.csv", index=False)

# --- visibility & RSF ---
vis_rows, rsf_rows = [], []
for mk, res in all_results.items():
    vr = res.get("visibility_rsf", {})
    for row in vr.get("visibility_table", []):
        vis_rows.append(dict(model=mk, **row))
    rsf_rows.append(dict(
        model=mk,
        rsf_boundary_pi_pred=vr.get("rsf_boundary_in_pi_pred"),
        rsf_boundary_pi_input=vr.get("rsf_boundary_in_pi_input"),
        rsf_ratio_input_over_pred=vr.get("rsf_ratio_input_over_pred"),
        rsf_factual_pred_in_pi_pred=vr.get("rsf_factual_pred_in_pi_pred"),
        rsf_factual_pred_in_pi_input=vr.get("rsf_factual_pred_in_pi_input"),
        rsf_random_in_pi_pred=vr.get("rsf_random_in_pi_pred"),
        rsf_random_in_pi_input=vr.get("rsf_random_in_pi_input"),
    ))
tab_vis = df_or_empty(vis_rows)
tab_rsf = df_or_empty(rsf_rows)
tab_vis.to_csv(RESULTS_DIR / "table_visibility.csv", index=False)
tab_rsf.to_csv(RESULTS_DIR / "table_rsf.csv", index=False)
print("\n=== VISIBILITY at m=64 ===")
if not tab_vis.empty:
    print(tab_vis[tab_vis["cutoff"]==64][
        ["model","boundary","factual_pred_mean","random_mean"]
    ].round(4).to_string(index=False))
print("\n=== RSF mechanism (boundary in Π_input vs Π_pred) ===")
if not tab_rsf.empty:
    print(tab_rsf[["model","rsf_boundary_pi_pred","rsf_boundary_pi_input",
                   "rsf_ratio_input_over_pred"]].round(4).to_string(index=False))

# --- fisher / hessian ---
fh_rows = []
for mk, res in all_results.items():
    for split, vals in res.get("fisher_hessian", {}).items():
        fh_rows.append(dict(model=mk, split=split, **vals))
tab_fh = df_or_empty(fh_rows)
tab_fh.to_csv(RESULTS_DIR / "table_fisher_hessian.csv", index=False)

# --- detection ---
det_rows = []
for mk, res in all_results.items():
    det_rows.extend(res.get("detection", []))
tab_det = df_or_empty(det_rows)
tab_det.to_csv(RESULTS_DIR / "table_detection.csv", index=False)
if not tab_det.empty:
    print("\n=== DETECTION AUROC (hall+imp vs factual) ===")
    sub = tab_det[tab_det["setup"]=="hall+imp_vs_fact"]
    print(sub[["model","auroc_probe","auroc_inv_maxprob","auroc_entropy"]]
          .round(4).to_string(index=False))

# --- random controls ---
rc_rows = []
for mk, res in all_results.items():
    rc_rows.extend(res.get("random_control", []))
tab_rc = df_or_empty(rc_rows)
tab_rc.to_csv(RESULTS_DIR / "table_random_control.csv", index=False)

# --- per-layer probe (for the AUROC-vs-depth figure) ---
prob_rows = []
for mk, res in all_results.items():
    for layer, m in res.get("probes", {}).get("per_layer", {}).items():
        prob_rows.append(dict(model=mk, layer=int(layer), **m))
tab_probe_depth = df_or_empty(prob_rows)
tab_probe_depth.to_csv(RESULTS_DIR / "table_probe_per_layer.csv", index=False)

print("\nall tables written to", RESULTS_DIR.resolve())


## 19. Plots

Quick-look figures for the paper. Saved to `results_causal/figs/`.


In [ ]:
import matplotlib.pyplot as plt
FIG_DIR = RESULTS_DIR / "figs"; FIG_DIR.mkdir(exist_ok=True)

# 1. Bypass precision matrix
if not tab_bypass.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    for ax, split in zip(axes, ["factual","impossible","hallucination"]):
        sub = tab_bypass[tab_bypass["split"]==split]
        for mk, g in sub.groupby("model"):
            ax.plot(g["gamma"], g["refusal_rate"], marker="o", label=mk)
        ax.set_title(f"Bypass refusal — {split}")
        ax.set_xlabel("γ"); ax.set_xscale("symlog", linthresh=1.0)
        ax.set_ylim(-0.02, 1.02)
    axes[0].set_ylabel("refusal rate")
    axes[-1].legend(loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=8)
    fig.tight_layout(); fig.savefig(FIG_DIR / "bypass_precision.png", dpi=150)
    plt.show()

# 2. Layer-wise probe accuracy
if not tab_probe_depth.empty:
    fig, ax = plt.subplots(figsize=(8, 5))
    for mk, g in tab_probe_depth.groupby("model"):
        ax.plot(g["layer"]/g["layer"].max(), g["in_domain_auroc"], label=mk, alpha=0.85)
    ax.set_xlabel("relative depth"); ax.set_ylabel("probe AUROC (factual vs impossible)")
    ax.set_ylim(0.5, 1.02); ax.legend(fontsize=8); ax.set_title("Layer-wise probe AUROC")
    fig.tight_layout(); fig.savefig(FIG_DIR / "probe_per_layer.png", dpi=150)
    plt.show()

# 3. Visibility comparison
if not tab_vis.empty:
    fig, ax = plt.subplots(figsize=(8, 5))
    sub = tab_vis[tab_vis["cutoff"]==64]
    x = np.arange(len(sub))
    w = 0.27
    ax.bar(x - w, sub["boundary"], w, label="boundary $b_L$")
    ax.bar(x,     sub["factual_pred_mean"], w, label="factual prediction")
    ax.bar(x + w, sub["random_mean"], w, label="random")
    ax.set_xticks(x); ax.set_xticklabels(sub["model"], rotation=45, ha="right")
    ax.set_ylabel("visibility at m=64"); ax.legend()
    fig.tight_layout(); fig.savefig(FIG_DIR / "visibility.png", dpi=150)
    plt.show()

# 4. RSF mechanism
if not tab_rsf.empty:
    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(tab_rsf)); w = 0.4
    ax.bar(x - w/2, tab_rsf["rsf_boundary_pi_pred"], w, label="$b_L$ in $\\Pi_{pred}$")
    ax.bar(x + w/2, tab_rsf["rsf_boundary_pi_input"], w, label="$b_L$ in $\\Pi_{input}$")
    ax.set_xticks(x); ax.set_xticklabels(tab_rsf["model"], rotation=45, ha="right")
    ax.set_ylabel("RSF"); ax.legend()
    ax.set_title("SVA mechanism test: boundary lives in input subspace, not prediction manifold")
    fig.tight_layout(); fig.savefig(FIG_DIR / "rsf_mechanism.png", dpi=150)
    plt.show()

# 5. Detection AUROC comparison
if not tab_det.empty:
    sub = tab_det[tab_det["setup"]=="hall+imp_vs_fact"]
    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(sub)); w = 0.27
    ax.bar(x - w, sub["auroc_probe"], w, label="probe (hidden state)")
    ax.bar(x,     sub["auroc_inv_maxprob"], w, label="1 − max prob")
    ax.bar(x + w, sub["auroc_entropy"], w, label="predictive entropy")
    ax.set_xticks(x); ax.set_xticklabels(sub["model"], rotation=45, ha="right")
    ax.set_ylabel("AUROC"); ax.legend()
    ax.set_title("Detection: probe vs output-confidence baselines")
    fig.tight_layout(); fig.savefig(FIG_DIR / "detection_auroc.png", dpi=150)
    plt.show()


## 20. Notes for using these results

**Key tables for the paper:**

- `table_bypass.csv` — the *bypass precision matrix*. The row to highlight in
  the paper is the γ where Hallucination/Impossible refusal saturates while
  Factual refusal stays low. That's the new headline number.
- `table_visibility.csv` — boundary vs factual-prediction vs random visibility
  at every cutoff. The claim in Section 4 of the paper is now numerical.
- `table_rsf.csv` — the SVA mechanism test. The prediction is
  `rsf_ratio_input_over_pred > 1` consistently. If it holds across all 8
  models, that's a clean mechanistic confirmation.
- `table_detection.csv` — practical utility: probe AUROC should beat
  `1 − max_prob` and entropy on the hall+imp vs factual setup.
- `table_steering.csv` and `table_repair.csv` — the original two tables,
  now with full sweeps.
- `table_random_control.csv` — random-direction baseline for boundary
  steering.
- `table_fisher_hessian.csv` — Fisher / Hessian per split, with the matched
  random-direction values available alongside.
- `table_probe_per_layer.csv` — feeds the layer-wise AUROC figure.

**Things to check after the first run:**

1. The uncertainty token printed for each tokenizer at the bypass step. If it
   resolves to something semantically off for a particular model (e.g. only
   the BPE fragment of `" un"`), swap in a better candidate by editing
   `UNCERTAIN_TOKEN_CANDIDATES`.
2. The intervention layer for the 8B/32B models is set heuristically. If the
   layer-wise probe AUROC curve peaks elsewhere, re-run with that layer for a
   tighter result.
3. If Qwen3-32B OOMs, set `MODELS['qwen3-32B']['load_in_4bit'] = True` and
   re-run only that key.
